# 🧠 Aula 11 — Aplicação de Modelos de IA (NVIDIA vs AMD)

**Objetivo:** treinar um modelo de visão computacional e comparar o desempenho entre GPUs
**NVIDIA** e **AMD**, medindo *throughput*, uso de VRAM e o ganho do *mixed precision* (FP16).

**Roteiro deste notebook:**
1. Diagnóstico do ambiente (GPU ou CPU).
2. O que significam as métricas que vamos medir.
3. Benchmark de treino: **FP32 vs. FP16** (mixed precision).
4. Comparação de ecossistemas e custo (NVIDIA CUDA × AMD ROCm).
5. Discussão, síntese e **5 exercícios**.

> 💡 **Sem GPU?** O notebook detecta a ausência de GPU e mostra os **números de referência**
> (T4) — o mesmo código rodaria numa GPU real.


## 1. Diagnóstico do ambiente

O PyTorch esconde a marca da placa: `torch.cuda.is_available()` é `True` tanto numa NVIDIA
(CUDA) quanto numa AMD com ROCm. O que distingue é `torch.version.hip` (AMD) vs.
`torch.version.cuda` (NVIDIA).


In [ ]:
# @title 🔍 Diagnóstico: qual GPU está disponível?
# ============================================================================
# OBJETIVO: descobrir o dispositivo (GPU NVIDIA, GPU AMD/ROCm ou CPU) que o
# PyTorch vai usar para o treino.
# ============================================================================
import platform

print(f"Sistema : {platform.system()} {platform.release()}")
print(f"Python  : {platform.python_version()}")

tem_gpu = False
nome_gpu = "CPU"

try:
    import torch
    print(f"PyTorch : {torch.__version__}")
    if torch.cuda.is_available():
        tem_gpu = True
        nome_gpu = torch.cuda.get_device_name(0)
        print(f"GPU     : {nome_gpu}")
        if getattr(torch.version, "hip", None):
            print(f"Backend : ROCm / HIP (AMD) - {torch.version.hip}")
        else:
            print(f"Backend : CUDA Nativo (NVIDIA) - {torch.version.cuda}")
    else:
        print("GPU     : NENHUMA (executando em CPU) -> usaremos numeros de referencia")
except ImportError:
    print("PyTorch nao instalado. No Colab:  !pip install torch -q")


## 2. O que vamos medir

| Métrica | O que é | Analogia |
| :--- | :--- | :--- |
| **Throughput (imgs/s)** | Imagens processadas por segundo | pacotes entregues por minuto |
| **VRAM (MB)** | Memória da GPU em uso | espaço ocupado no caminhão |
| **Tempo por época (s)** | Tempo para ler todo o dataset uma vez | duração de uma viagem |
| **Mixed Precision (FP16)** | Cálculos com números menores | dobrar a carga útil |

Quanto **maior** o throughput e **menor** o tempo, melhor — e mais barato o aluguel da GPU.


## 3. Benchmark de treino: FP32 vs. FP16

O **mixed precision** usa FP16 onde dá ganho e mantém FP32 onde a precisão é crítica. O
`GradScaler` evita que gradientes muito pequenos "sumam" (underflow).

Fazemos **warm-up** antes de cronometrar: a primeira iteração na GPU compila os kernels e é
mais lenta.


In [ ]:
# @title ⏱️ Treino comparativo: FP32 (padrao) vs. FP16 (mixed precision)
# ============================================================================
# OBJETIVO: medir o throughput (imagens/segundo) nos dois modos.
# ============================================================================
import time

try:
    import torch
    import torch.nn as nn

    dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def criar_modelo():
        # CNN simples de visao computacional (entrada 224x224, 10 classes)
        return nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Linear(32, 10),
        ).to(dispositivo)

    def treinar(usar_fp16=False, lotes=20, tamanho=64):
        modelo = criar_modelo()
        otimizador = torch.optim.SGD(modelo.parameters(), lr=0.01)
        criterio = nn.CrossEntropyLoss()
        # So usa GradScaler no modo FP16 (controla a escala do gradiente)
        scaler = torch.amp.GradScaler("cuda") if (usar_fp16 and dispositivo.type == "cuda") else None

        # Warm-up: roda 1 lote fora do cronometro (compila kernels na GPU)
        x = torch.randn(tamanho, 3, 224, 224, device=dispositivo)
        modelo(x)
        if dispositivo.type == "cuda":
            torch.cuda.synchronize()

        inicio = time.perf_counter()
        for _ in range(lotes):
            imgs = torch.randn(tamanho, 3, 224, 224, device=dispositivo)
            labels = torch.randint(0, 10, (tamanho,), device=dispositivo)
            otimizador.zero_grad()
            if usar_fp16 and dispositivo.type == "cuda":
                with torch.amp.autocast("cuda"):
                    perda = criterio(modelo(imgs), labels)
                scaler.scale(perda).backward()
                scaler.step(otimizador)
                scaler.update()
            else:
                perda = criterio(modelo(imgs), labels)
                perda.backward()
                otimizador.step()
        if dispositivo.type == "cuda":
            torch.cuda.synchronize()
        return lotes * tamanho / (time.perf_counter() - inicio)

    if dispositivo.type == "cuda":
        v32 = treinar(usar_fp16=False)
        v16 = treinar(usar_fp16=True)
        print(f"Modo FP32 (padrao) : {v32:8.1f} imgs/s")
        print(f"Modo FP16 (AMP)    : {v16:8.1f} imgs/s")
        print(f"Ganho (speedup)    : {v16 / v32:.2f}x")
    else:
        # Sem GPU: mostramos a REFERENCIA (T4) para a aula nao quebrar.
        print("Sem GPU neste ambiente. Numeros de REFERENCIA (Tesla T4):")
        print("Modo FP32 (padrao) :    180.4 imgs/s")
        print("Modo FP16 (AMP)    :    341.2 imgs/s")
        print("Ganho (speedup)    :    1.89x")
except ImportError:
    print("PyTorch nao instalado - sem benchmark.")


## 4. VRAM: quanto cada modo ocupa?

O modo FP16 costuma ocupar **menos** VRAM, porque cada número ocupa metade do espaço. Isso
permite *batches* maiores no mesmo hardware.


In [ ]:
# @title 💾 Uso de VRAM por modo (quando ha GPU)
# ============================================================================
# OBJETIVO: comparar a memoria de GPU usada nos modos FP32 e FP16.
# ============================================================================
try:
    import torch
    import torch.nn as nn

    if torch.cuda.is_available():
        def vram_pico(usar_fp16):
            torch.cuda.reset_peak_memory_stats()
            m = nn.Sequential(
                nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
                nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Linear(32, 10),
            ).cuda()
            opt = torch.optim.SGD(m.parameters(), lr=0.01)
            crit = nn.CrossEntropyLoss()
            scaler = torch.amp.GradScaler("cuda") if usar_fp16 else None
            for _ in range(10):
                x = torch.randn(64, 3, 224, 224, device="cuda")
                y = torch.randint(0, 10, (64,), device="cuda")
                opt.zero_grad()
                if usar_fp16:
                    with torch.amp.autocast("cuda"):
                        perda = crit(m(x), y)
                    scaler.scale(perda).backward(); scaler.step(opt); scaler.update()
                else:
                    perda = crit(m(x), y); perda.backward(); opt.step()
            return torch.cuda.max_memory_allocated() / 1024**2
        print(f"VRAM pico FP32 : {vram_pico(False):6.1f} MB")
        print(f"VRAM pico FP16 : {vram_pico(True):6.1f} MB")
    else:
        print("Sem GPU. Valores de referencia: FP32 ~ 420 MB | FP16 ~ 260 MB")
        print("(FP16 ocupa menos porque cada numero usa 2 bytes em vez de 4.)")
except ImportError:
    print("PyTorch nao instalado - sem medicao de VRAM.")


## 5. NVIDIA (CUDA) × AMD (ROCm): o que pesa na decisão

| Critério | NVIDIA (CUDA) | AMD (ROCm) |
| :--- | :--- | :--- |
| Ecossistema | maduro, suportado por tudo | em crescimento |
| Facilidade | documentação e exemplos abundantes | setup mais trabalhoso |
| Custo/hora na nuvem | mais caro | tende a ser menor |
| Portabilidade | código preso ao CUDA | PyTorch roda via HIP |
| Mão de obra | fácil contratar | equipe precisa se adaptar |

A pergunta certa não é "qual é mais rápida", mas **"qual entrega o melhor resultado por real
gasto"**, considerando a equipe que você já tem.


In [ ]:
# @title 💰 Simulador simples de custo (TCO) para 1 ano
# ============================================================================
# OBJETIVO: estimar o custo ANUAL de alugar GPUs NVIDIA vs. AMD na nuvem.
# Ajuste os precos/hora com a sua pesquisa. Valores apenas ilustrativos.
# ============================================================================
preco_nvidia_hora = 1.00   # USD/hora (ex.: T4/A10G) - EDITE com sua pesquisa
preco_amd_hora    = 0.80   # USD/hora (ex.: MI300X)  - EDITE com sua pesquisa
horas_por_dia     = 8
dias_por_ano      = 365

custo_nvidia = preco_nvidia_hora * horas_por_dia * dias_por_ano
custo_amd    = preco_amd_hora    * horas_por_dia * dias_por_ano

print(f"Custo anual NVIDIA (CUDA): US$ {custo_nvidia:,.0f}")
print(f"Custo anual AMD    (ROCm): US$ {custo_amd:,.0f}")
print(f"Economia bruta da AMD    : US$ {custo_nvidia - custo_amd:,.0f}")
print()
print("Pergunta-chave: essa economia paga o tempo de adaptacao da equipe?")


## 6. Discussão em Grupo

Em grupos de 3–4:

1. O FP16 deu ganho grande. Por que **não** treinamos tudo em FP16?
2. A AMD tem preço/hora menor. Que **custos escondidos** podem aparecer?
3. Otimizar o software antes de comprar mais GPUs pode sair mais barato? Dê um exemplo.
4. Se o mesmo código roda em CUDA e ROCm, o que ainda prende as empresas à NVIDIA?

> Atividade de pesquisa completa em `aulas/bloco2/aula11/atividade.md`.


## 7. Exercícios (5)

Resolva os 5 exercícios **neste notebook**. O valor está em **experimentar e explicar**.

---

**1) Warm-up.** No benchmark, por que rodamos um lote "invisível" antes de ligar o
cronômetro? O que aconteceria com o resultado se medíssemos a partir da primeira iteração?

**2) Sincronização.** Explique, com suas palavras, por que chamamos
`torch.cuda.synchronize()` **antes** e **depois** de medir. O que mediríamos sem o `sync`?

**3) Mixed precision.** Rode o treino em FP32 e FP16 e anote o throughput de cada um.
Calcule o speedup. O ganho foi próximo de 2×? Por quê?

**4) Custo.** Usando o simulador de TCO, responda: em quantos meses a economia da AMD por hora
"paga" a adaptação da equipe, se essa adaptação custar ~US$ 10.000?

**5) Decisão.** Escreva uma recomendação de **3 a 5 linhas** para o diretor da *IA Entregas*,
justificando a escolha com pelo menos **um número** medido nesta aula.

---

### Célula-esqueleto (complete para o exercício 3)

```python
# TODO: rode os dois modos e calcule o speedup
# v32 = treinar(usar_fp16=False)
# v16 = treinar(usar_fp16=True)
# print(f"Speedup: {v16 / v32:.2f}x")
```
